<a href="https://colab.research.google.com/github/liamcohenn/tp_laboratorio_futbol/blob/main/notebooks/tp_inferiores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trabajo Práctico Grupal - Laboratorio**

**Integrantes:** Liam Cohen, Renzo Bianco Maura, Federico Kurozwan y Milagros Leon

**Pregunta:** ¿Qué peso y qué rentabilidad tienen las ventas de jugadores formados en inferiores frente a las de jugadores comprados?

**Dataset:** Football Data from Transfermarkt (tablas transfers, players, clubs y competitions), subido al repositorio del grupo.

**Ligas analizadas:** Premier League (GB1), LaLiga (ES1), Ligue 1 (FR1), Primeira Liga (PO1), Eredivisie (NL1), Argentina (ARG1), México (MEX1) y Arabia Saudita (SA1).

# 3.1. Carga y exploración inicial

 ### Carga de datos



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/liamcohenn/tp_laboratorio_futbol/main/datos/raw/"

transfers = pd.read_csv(BASE + "transfers.csv.gz")
players = pd.read_csv(BASE + "players.csv.gz")
clubs = pd.read_csv(BASE + "clubs.csv.gz")
competitions = pd.read_csv(BASE + "competitions.csv.gz")

### Primeras filas de `transfers`

In [ ]:
transfers.head()

### Últimas filas de `transfers`

In [ ]:
transfers.tail()

### Cantidad de filas y columnas de `transfers`:

In [ ]:
transfers.shape

### Tipo de dato de cada columna de `transfers`:

In [ ]:
transfers.dtypes

### Valores nulos en `transfers`:

In [ ]:
transfers.isnull().sum()

### Filas duplicadas en `transfers`:

In [ ]:
transfers.duplicated().sum()

### Estadísticas descriptivas de `transfers`:

In [ ]:
transfers.describe()

#### Pasamos ahora con la tabla `players`
### Primeras filas de `players`:

In [ ]:
players.head()

### Últimas filas de `players`:

In [ ]:
players.tail()

### Cantidad de filas y columnas de `players`:

In [ ]:
players.shape

### Tipo de dato de cada columna de `players`:

In [ ]:
players.dtypes

### Valores nulos en `players`:

In [ ]:
players.isnull().sum()

### Filas duplicadas en `players`:

In [ ]:
players.duplicated().sum()

### Estadísticas descriptivas de `players`:

In [ ]:
players.describe()

## Comentarios sobre la primera parte del analisis


>

- **Tamaño:** `transfers` tiene 175.165 filas y 10 columnas; `players` tiene 50.149 filas y 26 columnas.


- **Duplicados:** no hay filas duplicadas ni en `transfers` ni en `players` (0 en las dos).

- **Fechas:** las transferencias van del 1 de julio de 1993 al 30 de junio de 2030. Hay 520 posteriores al 6 de julio de 2026, que son fechas futuras y las sacamos. Después de filtrar quedan 174.645 filas y la fecha más reciente es el 2 de julio de 2026.

 - **Valores nulos:** lo que más se nota es que a `transfers` le faltan muchos datos de dinero: `transfer_fee` tiene 61.526 nulos y `market_value_in_eur` tiene 68.160. Como no sabemos si esas transferencias fueron gratis o simplemente no se cargó el monto, para ingresos por ventas solo vamos a usar las filas que tienen `transfer_fee`. En `players` los nulos están sobre todo en datos de selección: `current_national_team_id` (46.889), `international_caps` e `international_goals` (30.793 cada una). También falta `agent_name` en 23.296 jugadores.

- **Valores llamativos:** en `transfer_fee` el máximo es de €222 millones, pero la mediana es €0, osea que la mayoría de los movimientos tiene un monto de cero, solo unos pocos son muy caros y no hay valores negativos. El valor de mercado más alto es €180 millones. En `players`, un jugador figura con 17 cm de altura, que no puede ser real y seguramente es un error de carga; la altura máxima es 210 cm.







## 3.2 Pregunta de investigación

**a) ¿Cuál es la pregunta?**
¿Qué peso y qué rentabilidad tienen las ventas de jugadores formados en inferiores frente a las de jugadores comprados, en los clubes de 8 ligas?

**b) ¿Por qué vale la pena responderla?**
Se entiende que formar jugadores en inferiores le sale mucho más barato a un club que comprarlos, ya que después los puede vender y ganar plata con eso. Saber cuánto pesan esas ventas sobre el total, y si dejan más ganancia que la venta de jugadores comprados, ayuda a entender si a un club le conviene invertir en sus inferiores o comprar jugadores ya formados.

**c) ¿Qué respuesta encontraron?**
completar al terminar el trabajo

#3.3. Transformaciones y resumen estadístico

#### **Transformación 1: pasar las fechas a formato fecha.** Las columnas `transfer_date` y `date_of_birth` están como texto. Las convertimos con `pd.to_datetime()` para poder restarlas y compararlas.

In [ ]:
transfers["transfer_date"] = pd.to_datetime(transfers["transfer_date"])
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"])

transfers.dtypes

#### **Transformación 2: sacar las transferencias posteriores al 6 de julio de 2026.** Son finales de préstamo programados, todavía no pasaron, así que no sirven para el análisis. Primero contamos cuántas hay (deberían ser 520) y después las eliminamos.

In [ ]:
fecha_corte = pd.to_datetime("2026-07-06")

print("Transferencias posteriores al corte:", (transfers["transfer_date"] > fecha_corte).sum())

transfers = transfers[transfers["transfer_date"] <= fecha_corte]

print("Fecha máxima ahora:", transfers["transfer_date"].max())
print("Filas que quedan:", transfers.shape[0])

#### **Transformación 3: calcular la edad del jugador en cada transferencia.** Pegamos la fecha de nacimiento de cada jugador a la tabla `transfers` (con `merge`, usando `player_id`) y calculamos la edad como la diferencia entre la fecha de la transferencia y la de nacimiento, en años. La vamos a necesitar en la Parte 2 para saber si un jugador entró al club antes de los 18.

In [ ]:
transfers = transfers.merge(players[["player_id", "date_of_birth"]], on="player_id", how="left")

transfers["edad"] = (transfers["transfer_date"] - transfers["date_of_birth"]).dt.days / 365.25

transfers[["player_name", "transfer_date", "date_of_birth", "edad"]].head()

#### **Transformación 4: lista de clubes de las 8 ligas.** Armamos la lista de ligas y nos quedamos con los clubes de la tabla `clubs` que juegan en alguna de ellas.

In [ ]:
ligas = ["GB1", "ES1", "FR1", "PO1", "NL1", "ARG1", "MEX1", "SA1"]

clubes_ligas = clubs[clubs["domestic_competition_id"].isin(ligas)]

clubes_ligas["domestic_competition_id"].value_counts()

Transformación 5: nombres de los clubes de las 8 ligas. La tabla clubs tiene los nombres oficiales largos (por ejemplo, “Futbol Club Barcelona”), pero en transfers aparecen los nombres cortos (“Barcelona”). Armamos un diccionario con el nombre corto de cada club, que vamos a usar para reconocer a sus equipos juveniles.

Asistencia de IA: las celdas de la Parte 2 se desarrollaron con asistencia de Claude. Prompt utilizado:[“Escribí en pandas una regla para clasificar las ventas de jugadores como canteranos o comprados, unificando los equipos juveniles con su club y sin contar los regresos de préstamo como compras”].

In [ ]:
import re

ids_ligas = set(clubes_ligas["club_id"])

salidas = transfers[transfers["from_club_id"].isin(ids_ligas)][["from_club_id", "from_club_name"]]
salidas.columns = ["club_id", "nombre"]
llegadas = transfers[transfers["to_club_id"].isin(ids_ligas)][["to_club_id", "to_club_name"]]
llegadas.columns = ["club_id", "nombre"]

nombre_club = pd.concat([salidas, llegadas]).groupby("club_id")["nombre"].agg(lambda s: s.value_counts().index[0])
print("Clubes de las 8 ligas:", len(nombre_club))